# Modelando o Banco com SQLAlchemy Core

---

## Contexto: desenhando o schema da DataVendas

O gerente de dados te passou o diagrama do banco e pediu para você **criar o schema** via código Python.  
"Por que código e não direto no banco?" 
Porque assim o schema entra no **controle de versão** (Git), fica documentado e é reproduzível em qualquer ambiente.

Nesta aula você vai:
- Criar tabelas com `MetaData` e `Table`
- Definir colunas com tipos e restrições realistas
- Usar `ForeignKey` para relacionar tabelas
- Inspecionar o banco criado
- Usar IA para gerar schemas a partir de descrições

---

## 1. Configuração

In [1]:
# Importações necessárias do SQLAlchemy para modelagem de tabelas
from pathlib import Path
from sqlalchemy import (
    create_engine, text, inspect,
    MetaData, Table, Column,
    Integer, String, DateTime, Numeric, ForeignKey, CheckConstraint, Boolean
)

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")

print("Pronto! ✅")

Pronto! ✅


## 2. MetaData — o catálogo do banco

O `MetaData` é um objeto que funciona como um **catálogo central**: ele registra todas as tabelas que você define.  
Quando você chamar `create_all()`, ele sabe exatamente o que criar no banco.

```
MetaData
  ├── tb_clientes
  ├── tb_pedidos
  ├── tb_produtos
  └── tb_itens_pedidos
```

In [2]:
# Cria o objeto MetaData, que é o catálogo central para registrar tabelas
metadata = MetaData()
print(f"MetaData criado: {metadata}")

MetaData criado: MetaData()


## 3. Criando as tabelas

### 3.1 Tabela de Clientes

Cada `Column` define um campo com seu tipo e suas regras de integridade:

| Parâmetro | O que garante |
|---|---|
| `primary_key=True` | Identificador único da linha |
| `nullable=False` | Campo obrigatório — não pode ficar vazio |
| `unique=True` | Não pode ter valores repetidos na coluna |

In [3]:
# Define a tabela de clientes com suas colunas e restrições
tb_clientes = Table(
    "tb_clientes",  # Nome da tabela no banco
    metadata,  # Registra no MetaData
    Column("id",            Integer,      primary_key=True),  # Chave primária, autoincremento
    Column("nome",          String(100),  nullable=False),    # Nome obrigatório, até 100 chars
    Column("email",         String(150),  nullable=False, unique=True),  # Email único e obrigatório
    Column("cidade",        String(80),   nullable=False),    # Cidade obrigatória
    Column("estado",        String(2),    nullable=False),    # Estado obrigatório (UF)
    Column("data_cadastro", DateTime,     nullable=False),    # Data de cadastro obrigatória
    extend_existing=True  # Permite redefinir se já existir no metadata
)

print("Tabela 'tb_clientes' definida ✅")

Tabela 'tb_clientes' definida ✅


### 3.2 Tabela de Produtos

In [4]:
# Define a tabela de produtos
tb_produtos = Table(
    "tb_produtos",
    metadata,
    Column("id",        Integer,       primary_key=True),  # ID único do produto
    Column("nome",      String(150),   nullable=False),    # Nome do produto
    Column("categoria", String(80),    nullable=False),    # Categoria do produto
    Column("preco_atual",     Numeric(10, 2), nullable=False),  # Preço com 2 casas decimais
    Column("ativo",     Boolean,        nullable=False, default=True),  # Se o produto está ativo
    extend_existing=True
)

print("Tabela 'tb_produtos' definida ✅")

Tabela 'tb_produtos' definida ✅


> 💡 **Por que `Numeric` e não `Float`?**  
> `Float` é impreciso para dinheiro: `0.1 + 0.2 = 0.30000000000000004`.  
> `Numeric(10, 2)` garante exatidão decimal: use sempre para valores monetários.

### 3.3 Tabela de Pedidos

Aqui aparecem dois conceitos importantes:
- **`ForeignKey`**: conecta um pedido a um cliente (relação N:1)
- **`CheckConstraint`**: garante que o `status` só aceite valores do domínio definido

In [5]:
# Define a tabela de pedidos com foreign key e check constraint
tb_pedidos = Table(
    "tb_pedidos",
    metadata,
    Column("id",          Integer,                               primary_key=True),  # ID do pedido
    Column("cliente_id",  Integer, ForeignKey("tb_clientes.id"), nullable=False),  # FK para clientes
    Column("data_pedido", DateTime,                              nullable=False),   # Data do pedido
    Column("valor_total", Numeric(10, 2),                        nullable=False),   # Valor total
    Column("status",      String(20),                            nullable=False),   # Status do pedido

    # Constraint para validar valores de status
    CheckConstraint(
        "status IN ('Criado', 'Pago', 'Enviado', 'Cancelado')",
        name="ck_status_pedido"
    ),
    extend_existing=True
)

print("Tabela 'tb_pedidos' definida ✅")

Tabela 'tb_pedidos' definida ✅


### 3.4 Tabela de Itens do Pedido

Esta tabela representa a relação **N:N** entre pedidos e produtos.  
É a tabela associativa clássica: um pedido tem vários produtos, e um produto pode estar em vários pedidos.

In [6]:
# Define a tabela associativa para itens dos pedidos (relação N:N)
tb_itens_pedidos = Table(
    "tb_itens_pedidos",
    metadata,
    Column("id",          Integer,                               primary_key=True),  # ID do item
    Column("pedido_id",   Integer, ForeignKey("tb_pedidos.id"),  nullable=False),  # FK para pedidos
    Column("produto_id",  Integer, ForeignKey("tb_produtos.id"), nullable=False),  # FK para produtos
    Column("quantidade",  Integer,                               nullable=False),   # Quantidade comprada
    Column("preco_venda",  Numeric(10, 2),                        nullable=False),   # Preço na venda
    extend_existing=True
)

print("Tabela 'tb_itens_pedidos' definida ✅")

Tabela 'tb_itens_pedidos' definida ✅


## 4. Criando as tabelas no banco

Até agora, tudo existia apenas **em memória** (no objeto `metadata`).  
O `create_all` envia os comandos `CREATE TABLE` para o banco de verdade.

> `create_all` é seguro para rodar múltiplas vezes: ele só cria tabelas que **ainda não existem**.

In [7]:
# Cria todas as tabelas definidas no MetaData no banco de dados
metadata.create_all(engine)
print("Todas as tabelas criadas no banco! ✅")

Todas as tabelas criadas no banco! ✅


## 5. Inspecionando o que foi criado

No trabalho, o `Inspector` é muito útil para **explorar bancos legados** que você não criou e para confirmar que o schema está do jeito que você espera.

In [8]:
# Cria um inspetor para examinar a estrutura do banco
inspector = inspect(engine)

# Itera sobre todas as tabelas do banco
for tabela in inspector.get_table_names():
    print(f"\nTabela: {tabela}")
    print(f"   {'Coluna':<25} {'Tipo':<20} {'Nullable':<10}")
    print(f"   {'-'*55}")
    # Para cada tabela, lista suas colunas com detalhes
    for col in inspector.get_columns(tabela):
        print(f"   {col['name']:<25} {str(col['type']):<20} {str(col['nullable']):<10}")

    # Verifica se há foreign keys e as lista
    fks = inspector.get_foreign_keys(tabela)
    if fks:
        print(f"   🔗 Foreign Keys:")
        for fk in fks:
            print(f"      {fk['constrained_columns']} → {fk['referred_table']}.{fk['referred_columns']}")


Tabela: tb_clientes
   Coluna                    Tipo                 Nullable  
   -------------------------------------------------------
   id                        INTEGER              False     
   nome                      VARCHAR(100)         False     
   email                     VARCHAR(150)         False     
   cidade                    VARCHAR(80)          False     
   estado                    VARCHAR(2)           False     
   data_cadastro             DATETIME             False     

Tabela: tb_itens_pedidos
   Coluna                    Tipo                 Nullable  
   -------------------------------------------------------
   id                        INTEGER              False     
   pedido_id                 INTEGER              False     
   produto_id                INTEGER              False     
   quantidade                INTEGER              False     
   preco_venda               NUMERIC(10, 2)       False     
   🔗 Foreign Keys:
      ['pedido_id'] → t

---

## Exercício: Usando IA para isso

Gerar schemas a partir de uma descrição em linguagem natural é um dos melhores usos de IA no dia a dia de dados.

**Prompt para gerar um schema completo:**
```
Preciso criar as tabelas de um sistema de RH no SQLAlchemy Core (Python).
As entidades são: Departamento, Funcionário e Cargo.
- Departamento: id, nome, centro_de_custo
- Funcionário: id, nome, email (único), data_admissão, salário (monetário), 
  id_departamento (FK), id_cargo (FK), ativo (boolean)
- Cargo: id, titulo, nivel (júnior/pleno/sênior — use CheckConstraint)
Use tipos adequados para valores monetários e datas.
```
---

### Resposta:Código gerado pelo Copilot

In [ ]:
# Código gerado para o sistema de RH usando SQLAlchemy Core

from sqlalchemy import (
    MetaData, Table, Column, Integer, String, DateTime, Numeric, 
    ForeignKey, CheckConstraint, Boolean
)

# Cria o MetaData
metadata_rh = MetaData()

# Tabela Departamento
tb_departamentos = Table(
    "tb_departamentos",
    metadata_rh,
    Column("id", Integer, primary_key=True),
    Column("nome", String(100), nullable=False),
    Column("centro_de_custo", String(50), nullable=False),
    extend_existing=True
)

# Tabela Cargo
tb_cargos = Table(
    "tb_cargos",
    metadata_rh,
    Column("id", Integer, primary_key=True),
    Column("titulo", String(100), nullable=False),
    Column("nivel", String(20), nullable=False),
    CheckConstraint(
        "nivel IN ('júnior', 'pleno', 'sênior')",
        name="ck_nivel_cargo"
    ),
    extend_existing=True
)

# Tabela Funcionário
tb_funcionarios = Table(
    "tb_funcionarios",
    metadata_rh,
    Column("id", Integer, primary_key=True),
    Column("nome", String(150), nullable=False),
    Column("email", String(150), nullable=False, unique=True),
    Column("data_admissao", DateTime, nullable=False),
    Column("salario", Numeric(12, 2), nullable=False),  # Salário monetário com 2 casas decimais
    Column("id_departamento", Integer, ForeignKey("tb_departamentos.id"), nullable=False),
    Column("id_cargo", Integer, ForeignKey("tb_cargos.id"), nullable=False),
    Column("ativo", Boolean, nullable=False, default=True),
    extend_existing=True
)

# Para criar as tabelas no banco:
# metadata_rh.create_all(engine)
# print("Tabelas do RH criadas!")

# Exemplo de inspeção:
# from sqlalchemy import inspect
# inspector = inspect(engine)
# for table in inspector.get_table_names():
#     if table.startswith('tb_'):
#         print(f"Tabela: {table}")